In [1]:
!pip list

Package                       Version
----------------------------- ------------
accelerate                    1.13.0
aiohappyeyeballs              2.6.1
aiohttp                       3.13.5
aiosignal                     1.4.0
annotated-doc                 0.0.4
annotated-types               0.7.0
anyio                         4.13.0
asttokens                     3.0.1
async-timeout                 5.0.1
attrs                         26.1.0
av                            17.0.1
bitsandbytes                  0.49.2
certifi                       2026.4.22
charset-normalizer            3.4.7
click                         8.3.3
comm                          0.2.3
contourpy                     1.3.2
cuda-bindings                 12.9.4
cuda-pathfinder               1.5.3
cut-cross-entropy             25.1.1
cycler                        0.12.1
datasets                      4.3.0
debugpy                       1.8.20
decorator                     5.2.1
diffusers                     0.37.1
dill

In [1]:
import sys
print(sys.executable)

# 1. Cleanup broken/mixed installs
!{sys.executable} -m pip uninstall -y torch torchvision torchaudio triton xformers bitsandbytes unsloth unsloth_zoo transformers

# 2. Upgrade installer tools
!{sys.executable} -m pip install -U pip uv

# 3. Install Unsloth + compatible Torch stack
!{sys.executable} -m uv pip install --reinstall unsloth --torch-backend=cu128

# 4. Install normal utilities
!{sys.executable} -m pip install matplotlib tiktoken pillow transformers_stream_generator h5py scikit-learn trackio

/home/annaw/cmsc472final-venv/bin/python
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
Found existing installation: xformers 0.0.35
Uninstalling xformers-0.0.35:
  Successfully uninstalled xformers-0.0.35
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: unsloth 2026.4.8
Uninstalling unsloth-2026.4.8:
  Successfully uninstalled unsloth-2026.4.8
Found existing installation: unsloth_zoo 2026.4.9
Uninstalling unsloth_zoo-2026.4.9:
  Successfully uninstalled unsloth_zoo-2026.4.9
Found existing installation: transformers 5.5.0
Uninstalling transformers-5.

In [2]:
!pip install qwen_vl_utils

In [3]:
import os

os.environ["HF_HOME"] = f"/home/annaw/scratch/hf-cache"
os.environ["TRANSFORMERS_CACHE"] = f"/home/annaw/scratch/hf-cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_CACHE"] = "/home/annaw/scratch/hf-cache"

In [4]:
#from unsloth import FastLanguageModel
import torch
import numpy
torch.cuda.is_available()

from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from transformers.generation import GenerationConfig
fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

"""model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    local_files_only=True,
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)"""
"""model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/home/annaw/scratch/hf-cache/models--unsloth--Qwen3-1.7B/snapshots/6262b50d6c1f8ee5e4ac750d710c33603bfc2a0c",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    local_files_only=True,
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)"""


model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct", torch_dtype="auto", device_map="auto", local_files_only=True,
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")


/home/annaw/cmsc472final-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Loading weights: 100%|██████████| 730/730 [00:26<00:00, 27.26it/s]


In [5]:
#source: https://huggingface.co/learn/cookbook/fine_tuning_vlm_trl
from qwen_vl_utils import process_vision_info

def generate_text_from_sample(model, processor, sample, max_new_tokens=1024, device="cuda"):
    # Prepare the text input by applying the chat template
    text_input = processor.apply_chat_template(
        sample['questions'][1:2],  # Use the sample without the system message
        tokenize=False,
        add_generation_prompt=True
    )

    # Process the visual input from the sample
    image_inputs, _ = process_vision_info(sample['images'])

    # Prepare the inputs for the model
    model_inputs = processor(
        text=[text_input],
        images=image_inputs,
        return_tensors="pt",
    ).to(device)  # Move inputs to the specified device

    # Generate text with the model
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Trim the generated ids to remove the input ids
    trimmed_generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decode the output text
    output_text = processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text[0]

The video I saw talked about LoRA adapters and I think its a technique for fine tuning.
If you fine tune a model based on a specific task it will probably lose its original weights and logic that made it good to begin with so this way you don't touch any of the pre-trained weights but you make these adapters with change the way the weights act and that's what we're tuning

https://www.youtube.com/watch?v=BJgjYhJf7h4

https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(14B)-Reasoning-Conversational.ipynb#scrollTo=736e7270e227

In [6]:
from pprint import pprint
import h5py
import json
from sklearn.model_selection import train_test_split
with open('train.json', 'r') as file:
    currentFileData = json.load(file)
dataset = []
images = h5py.File('dataset.h5', 'r')
print(images.keys())
dset = images['images']
for i, element in enumerate(currentFileData):
    accepeted_answers =set()
    for answers in element["answers"]:
        if answers['answer_confidence'] == "yes":
            accepeted_answers.add(answers['answer'])
    dataset.append({"question": element["question"], "answers": list(accepeted_answers), 'image': i})
    
train,test = train_test_split(
    dataset, test_size=0.2, random_state=42)

<KeysViewHDF5 ['images']>


In [7]:
from peft import LoraConfig

# Configure LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

In [8]:
from trl import SFTConfig

# Configure training arguments
training_args = SFTConfig(
    output_dir="qwen2-7b-instruct-trl-sft-ChartQA",  # Directory to save the model
    num_train_epochs=3,  # Number of training epochs
    per_device_train_batch_size=4,  # Batch size for training
    per_device_eval_batch_size=4,  # Batch size for evaluation
    gradient_accumulation_steps=8,  # Steps to accumulate gradients
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Options for gradient checkpointing
    max_length=None,
    # Optimizer and scheduler settings
    optim="adamw_torch_fused",  # Optimizer type
    learning_rate=2e-4,  # Learning rate for training
    # Logging and evaluation
    logging_steps=10,  # Steps interval for logging
    eval_steps=10,  # Steps interval for evaluation
    eval_strategy="steps",  # Strategy for evaluation
    save_strategy="steps",  # Strategy for saving the model
    save_steps=20,  # Steps interval for saving
    # Mixed precision and gradient settings
    bf16=True,  # Use bfloat16 precision
    max_grad_norm=0.3,  # Maximum norm for gradient clipping
    warmup_ratio=0.03,  # Ratio of total steps for warmup
    # Hub and reporting
    #push_to_hub=True,  # Whether to push model to Hugging Face Hub
    #report_to="trackio",  # Reporting tool for tracking metrics
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    dataloader_num_workers=0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
from datasets import Features, Sequence, Value, Image, Dataset

records = []

for i, sample in enumerate(train):

    records.append({
        "question": sample["question"],
        "answers": list(sample["answers"]),
        "image": i,
    })

    
train_dataset = Dataset.from_list(records)

records = []

for i, sample in enumerate(test):

    records.append({
        "question": sample["question"],
        "answers": list(sample["answers"]),
        "image": i,
    })
    
test_dataset = Dataset.from_list(records)

In [10]:
import h5py
from PIL import Image
class MyDataLoader:
    def __init__(self, processor, h5_path):
        self.processor = processor
        self.h5_path = h5_path
        self._h5 = None
    
    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __call__(self, examples):
        h5 = self._get_h5()
        dset = h5["images"]

        messages = []
        images = []
        
        for ex in examples:
            img_arr = dset[ex["image"]]
            image = Image.fromarray(img_arr)
            images.append(image)
            message = ([
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": ex["question"]},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": ex["answers"][0]},
                    ],
                },
            ])
            messages.append(message)
            
        texts = self.processor.apply_chat_template(messages, tokenizer=False, add_gereative_prompt=False)
        batch = self.processor(text=texts, images=images, return_tensors="pt", padding=True)
        
        batch["labels"] = batch["input_ids"].clone()
        return batch


In [ ]:
from trl import SFTTrainer

data_loader = MyDataLoader(
    processor=processor,
    h5_path="dataset.h5",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=data_loader,
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` hav

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,10.506876,10.426679,3.511414,18649.000000,0.233251
20,10.288875,10.034245,3.716332,37446.000000,0.272904
30,9.620264,9.099581,4.303596,56449.000000,0.316793
40,8.321361,7.062882,5.175443,75460.000000,0.348790
50,6.087571,5.405591,5.092513,94317.000000,0.400268
60,5.251903,5.128975,4.639249,113110.000000,0.404032
70,5.191624,5.025937,4.511389,132208.000000,0.406189
80,5.004671,4.955287,4.420562,151086.000000,0.408051
90,4.927863,4.811493,4.380583,169914.000000,0.408195
100,4.650573,4.718463,4.206282,188695.000000,0.424491


Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

...

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i

Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Kwargs passed to `processor.__call__` have to be i